# Lightweight ecommerce recommendations

Load real data as `df`. Read RESEARCH_REVIEW.md and METHODS_V2.md before interpreting the results. This workflow evaluates a categorical intervention policy; product IDs are output keys.

In [ ]:
# Locate root when running inside notebooks/.
from pathlib import Path
import sys
REPO_ROOT = next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'causal_dml.py').exists())
sys.path.insert(0,str(REPO_ROOT))
import os
os.chdir(REPO_ROOT)  # historical example cells use repository-relative paths

In [ ]:
%pip install -r requirements-notebooks.txt

In [ ]:
from pathlib import Path
import pandas as pd
from causal_dml import LightweightDRLearner, CustomCausalDML, FinalStageConfig
from recommendation_pipeline import RecommendationConfig, run_recommendations

In [ ]:
# df = pd.read_parquet('/path/to/actual_exposures.parquet')
if 'df' not in globals():
    raise RuntimeError('Load your actual attributed and mature exposure dataframe as df.')
print(df.shape)

In [ ]:
cfg = RecommendationConfig(
    outcome_col='revenue', group_col='user_id',
    x_cols=['data_definition', 'industry'],
    report_cols=['data_definition', 'industry'],
    w_cols=['day', 'week', 'month', 'device', 'channel'],
    lead_col='lead_time', discount_col='discount_percentage',
    discount_input_is_percent=True, max_discount=.45, max_lead=30,
    reference='0d|0%', log1p_outcome=False,
    min_action_rows=100, min_cohort_action_rows=100,
    propensity_floor=.001, min_policy_propensity=.01,
    calibrate=False,
    product_col=None,       # set to 'product_id' when available; kept as output key
    embedding_cols=[],      # optional frozen numeric product features; at most 128
    action_costs={},         # known deterministic costs in revenue units, by exact action label
    final_candidates=[
        FinalStageConfig(n_estimators=80, num_leaves=7, max_depth=3, min_child_samples=200),
        FinalStageConfig(n_estimators=120, num_leaves=15, max_depth=4, min_child_samples=300),
    ],
    scenarios=[{'name':'all', 'filters':{}},
               {'name':'lab_water', 'filters':{'data_definition':['lab_water'], 'industry':[]}}],
)
# To estimate carousel exposure instead of joint lead/discount actions:
# cfg.treatment_col='carousel_seen'
# cfg.categories=[0,1]
# cfg.reference=0
# cfg.known_propensities=[.5,.5]  # only if this was the actual constant randomization
# Review pretreatment X/W for this intervention.

## Fit and evaluate

Defaults use four disjoint user splits. Random groups do not establish future-time generalization. For a time design, supply externally purged `split_indices` for train, selection, calibration and test, indexing the eligible prepared frame. Outcome maturity and causal attribution must be validated before this cell.

In [ ]:
out=Path('recommendation_outputs')
fitted, report=run_recommendations(df,cfg,out)
report

In [ ]:
display(pd.read_csv(out/'preflight_actions.csv'))
display(pd.read_json(out/'validation_selection.json'))
report['policy_evaluation']

In [ ]:
display(pd.read_csv(out/'recommendations_test_population.csv').head(20))
display(pd.read_csv(out/'scenario_effects.csv'))
from IPython.display import Image
if (out/'cohort_heatmap.png').exists():
    display(Image(filename=str(out/'cohort_heatmap.png')))

## New eligible opportunities

For new data, use the fitted encoders and estimator. Apply the same cohort support/availability rules as the application pipeline. The bare estimator only enforces its propensity mask unless an `eligible` matrix is supplied. Do not reinterpret the observed test outcomes as training data to improve this reported policy.

In [ ]:
# current_opportunities must be your current eligible population with X/W features.
# x,w=fitted.encode(current_opportunities)
# recommendations=fitted.estimator.recommend(x,W=w,
#     costs=[0.0]*len(fitted.estimator.categories_),
#     min_propensity=cfg.min_policy_propensity, eligible=your_eligibility_matrix)

## Interpretation

The policy value report compares the fixed recommended action with the reference action on independent test users. The approximate interval is for that value contrast, not for individual CATEs. Read overlap, clipping and ESS fields before interpreting it. There is no hidden-confounding guarantee, no exact budget optimizer, and no validated effect for each rare product. Original revenue is used by default; log effects have a different geometric interpretation.